# Stage 4 — Baseline + Merge + Experiments


## 1. Load data

In [ ]:

import pandas as pd
from pathlib import Path

ROOT = Path("../")

emotion_path = ROOT / "datasets/emotion_annotated/metadata/pilot_video_emotion_features.csv"
manifest_path = ROOT / "exports/pilot/csv/pilot_manifest_export.csv"
detector_path = ROOT / "datasets/detector_processed/pilot_detector_scores.csv"

emotion_df = pd.read_csv(emotion_path)
manifest_df = pd.read_csv(manifest_path)
detector_df = pd.read_csv(detector_path)

print(len(emotion_df), len(manifest_df), len(detector_df))


## 2. Merge

In [ ]:

# Drop duplicate columns from emotion_df before merging
emotion_drop = [c for c in emotion_df.columns if c in manifest_df.columns and c != "video_id"]
df = manifest_df.merge(emotion_df.drop(columns=emotion_drop), on="video_id")

# Only bring score columns from detector_df to avoid further duplicates
detector_cols = ["video_id", "detector_score", "detector_pred"]
df = df.merge(detector_df[detector_cols], on="video_id")

# Encode label: "fake" → 1, "real" → 0
df["label"] = (df["label"].str.lower() == "fake").astype(int)

print("Shape:", df.shape)
print("Label distribution:\n", df["label"].value_counts())
df.head()


## 3. Metrics (baseline)

In [ ]:

import numpy as np
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score

y_true = df["label"].values
y_score = df["detector_score"].values
y_pred = (y_score >= 0.5).astype(int)

auc = roc_auc_score(y_true, y_score)
acc = accuracy_score(y_true, y_pred)
f1  = f1_score(y_true, y_pred)

# Bootstrap 95% CI for AUC
rng = np.random.default_rng(42)
boot_aucs = [
    roc_auc_score(y_true[idx := rng.integers(0, len(y_true), len(y_true))], y_score[idx])
    for _ in range(2000)
]
auc_lo, auc_hi = np.percentile(boot_aucs, [2.5, 97.5])

print(f"AUC : {auc:.4f}  (95% CI {auc_lo:.4f}–{auc_hi:.4f})")
print(f"ACC : {acc:.4f}")
print(f"F1  : {f1:.4f}")


## 4. Emotion analysis

In [ ]:

df["arousal_bin"] = pd.qcut(df["mean_arousal"], 3, labels=["low", "mid", "high"])

rng = np.random.default_rng(42)

print(f"{'Bin':<6}  {'AUC':>6}  {'95% CI':>18}  {'n':>4}")
print("-" * 42)
for g in ["low", "mid", "high"]:
    sub = df[df["arousal_bin"] == g]
    yt, ys = sub["label"].values, sub["detector_score"].values
    auc = roc_auc_score(yt, ys)
    boot = [
        roc_auc_score(yt[idx := rng.integers(0, len(yt), len(yt))], ys[idx])
        for _ in range(2000)
    ]
    lo, hi = np.percentile(boot, [2.5, 97.5])
    print(f"{str(g):<6}  {auc:.4f}  ({lo:.4f} – {hi:.4f})  {len(sub):>4}")


## 5. Fusion model

In [ ]:

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

features = [
    "detector_score",
    "mean_arousal",
    "std_arousal",
    "mean_valence",
    "emotion_entropy",
    "transition_rate",
]

X = df[features].fillna(0).values
y = df["label"].values

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fusion_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000)),
])

fusion_proba = cross_val_predict(fusion_pipe, X, y, cv=cv, method="predict_proba")[:, 1]
fusion_auc = roc_auc_score(y, fusion_proba)

rng = np.random.default_rng(42)
boot = [
    roc_auc_score(y[idx := rng.integers(0, len(y), len(y))], fusion_proba[idx])
    for _ in range(2000)
]
lo, hi = np.percentile(boot, [2.5, 97.5])

print(f"Fusion AUC (5-fold CV): {fusion_auc:.4f}  (95% CI {lo:.4f}–{hi:.4f})")


## 6. Compare

In [ ]:

# ── Detector baseline (no model needed — direct scores) ──────────────────────
baseline_auc = roc_auc_score(y, df["detector_score"].values)

# ── Emotion-only (5-fold CV, same pipeline) ───────────────────────────────────
emotion_features = [
    "mean_arousal", "std_arousal", "mean_valence",
    "emotion_entropy", "transition_rate",
]
X_emo = df[emotion_features].fillna(0).values

emotion_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000)),
])
emotion_proba = cross_val_predict(emotion_pipe, X_emo, y, cv=cv, method="predict_proba")[:, 1]
emotion_auc = roc_auc_score(y, emotion_proba)

rng2 = np.random.default_rng(42)
def _boot_auc(yt, ys, n=2000):
    rng_ = np.random.default_rng(42)
    b = [roc_auc_score(yt[i := rng_.integers(0, len(yt), len(yt))], ys[i]) for _ in range(n)]
    return np.percentile(b, [2.5, 97.5])

b_lo, b_hi = _boot_auc(y, df["detector_score"].values)
e_lo, e_hi = _boot_auc(y, emotion_proba)
f_lo, f_hi = lo, hi  # from fusion cell above

print(f"{'Model':<20}  {'AUC':>6}  {'95% CI':>18}")
print("-" * 50)
print(f"{'Detector only':<20}  {baseline_auc:.4f}  ({b_lo:.4f} – {b_hi:.4f})")
print(f"{'Emotion only (CV)':<20}  {emotion_auc:.4f}  ({e_lo:.4f} – {e_hi:.4f})")
print(f"{'Fusion (CV)':<20}  {fusion_auc:.4f}  ({f_lo:.4f} – {f_hi:.4f})")
